In [ ]:
import os
import re
import torch
import pandas as pd
from datetime import datetime
from typing import List, Set

# --- LangChain Imports ---
from langchain_community.document_loaders import WebBaseLoader, Docx2txtLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document 

# --- Helper Library for RTF ---
from striprtf.striprtf import rtf_to_text

# --- Transformer Imports ---
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# ================= CONFIGURATION =================
CONFIG = {
    "mode": "title",  
    "main_model_id": "Qwen/Qwen3-4B-Thinking-2507",
    "summ_model_id": "slovak-nlp/mistral-sk-7b", 
    "emb_model_id": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "input_dir": "./Input",
    "output_dir": "./Output",
    "rag_dir": "./RAG_store",
    "thesaurus_path": "./Thesaurus/SK_Local_Thesaurus.xlsx",
    "thesaurus_col": "slovak_terms",
    "stopwords_path": "./Input/stopword_dictionary.txt",
    "allowed_domains": ["slov-lex.sk", "justice.gov.sk", "najpravo.sk", "zakonypreludi.sk"],
    "chunk_size": 1200,
    "chunk_overlap": 200,
    "top_k": 4
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["rag_dir"], exist_ok=True)

# ================= 1. CUSTOM RTF LOADER =================
class SimpleRTFLoader:
    def __init__(self, file_path):
        self.file_path = file_path

    def load(self) -> List[Document]:
        try:
            with open(self.file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                text = rtf_to_text(content)
                return [Document(page_content=text, metadata={"source": self.file_path})]
        except Exception as e:
            print(f"[WARN] RTF parsing failed for {self.file_path}: {e}")
            return []

# ================= 2. HELPERS =================
def load_stopwords(path) -> Set[str]:
    if not os.path.exists(path): return set()
    try:
        with open(path, "r", encoding="utf-8") as f:
            return {line.strip().lower() for line in f if line.strip()}
    except: return set()

def load_thesaurus(path, col_name, stopwords) -> List[str]:
    if not os.path.exists(path): return []
    try:
        df = pd.read_excel(path, engine="openpyxl")
        if col_name not in df.columns: return []
        raw = df[col_name].dropna().astype(str).tolist()
        terms = []
        seen = set()
        for cell in raw:
            parts = re.split(r'[,\n;]+', cell)
            for t in parts:
                t = t.strip()
                if len(t) > 1 and t.lower() not in stopwords and t not in seen:
                    seen.add(t)
                    terms.append(t)
        print(f"[THESAURUS] Loaded {len(terms)} terms.")
        return terms
    except: return []

def match_thesaurus_terms(text: str, all_terms: List[str], limit=10) -> str:
    if not text or not all_terms: return ""
    hits = {}
    for t in all_terms:
        if re.search(r"\b" + re.escape(t) + r"\b", text, re.IGNORECASE):
            hits[t] = len(t)
    sorted_hits = sorted(hits.keys(), key=lambda x: hits[x], reverse=True)
    return ", ".join(sorted_hits[:limit])

# ================= 3. MODEL LOADERS =================
def load_cpu_model(model_id):
    print(f"[CPU] Loading {model_id}...")
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            device_map="cpu", 
            torch_dtype=torch.float32, 
            low_cpu_mem_usage=True
        )
        return pipeline("text-generation", model=model, tokenizer=tok)
    except Exception as e:
        print(f"[ERROR] CPU Load Failed: {e}")
        return None

def load_gpu_model(model_id):
    print(f"[GPU] Loading {model_id} on RTX 4070 Super...")
    try:
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)        
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="cuda:0",
            trust_remote_code=True,
            torch_dtype=torch.float16, # Standard safe dtype for 4070
            low_cpu_mem_usage=True
        )
        return pipeline("text-generation", model=model, tokenizer=tok)
    except Exception as e:
        print(f"[ERROR] GPU Load Failed: {e}")
        print("Tip: If running OOM on 12GB VRAM, switch to the 4B model option in CONFIG.")
        return None

# ================= 4. RAG MANAGER =================
class RAGManager:
    def __init__(self):
        print(f"[EMB] Loading Embeddings on CPU...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=CONFIG["emb_model_id"],
            model_kwargs={"device": "cpu"}
        )
        self.vector_store = Chroma(
            collection_name="sk_legal_docs",
            persist_directory=CONFIG["rag_dir"],
            embedding_function=self.embeddings
        )
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=CONFIG["chunk_size"], chunk_overlap=CONFIG["chunk_overlap"]
        )

    def add_text(self, text, metadata):
        docs = self.splitter.create_documents([text], metadatas=[metadata])
        if docs: self.vector_store.add_documents(docs)

    def search(self, query):
        docs = self.vector_store.similarity_search(query, k=CONFIG["top_k"])
        return "\n\n".join([d.page_content for d in docs])

    def ingest_web(self, urls):
        valid = [u for u in urls if any(d in u for d in CONFIG["allowed_domains"])]
        if not valid: return
        try:
            loader = WebBaseLoader(valid)
            docs = loader.load()
            split_docs = self.splitter.split_documents(docs)
            self.vector_store.add_documents(split_docs)
            print(f"[WEB] Ingested {len(split_docs)} chunks.")
        except: pass

# ================= 5. PIPELINE LOGIC =================
def process_file(filepath, rag_manager, summarizer, generator, stopwords, thesaurus_terms):
    filename = os.path.basename(filepath)
    print(f"--- Processing: {filename} ---")
    
    try:
        if filepath.endswith(".docx"):
            loader = Docx2txtLoader(filepath)
        elif filepath.endswith(".rtf"):
            loader = SimpleRTFLoader(filepath)
        else:
            loader = TextLoader(filepath, encoding="utf-8")
        
        raw_docs = loader.load()
        text = "\n".join([d.page_content for d in raw_docs])
    except Exception as e:
        print(f"[ERR] File load failed: {e}")
        return None

    if not text.strip(): return None

    rag_manager.add_text(text, {"file": filename})

    # Summarize (CPU)
    summ_input = text[:12000]
    prompt_summ = (f"Zhrň nasledovný text do stručného odstavca (max 5 viet). Píš po slovensky.\n\n"
                    f"TEXT:\n{summ_input}\n\nZHRNUTIE:")
    try:
        # do_sample=True enables creativity (temperature)
        summ_res = summarizer(prompt_summ, max_new_tokens=150, do_sample=True, temperature=0.2)
        summary = summ_res[0]["generated_text"].split("ZHRNUTIE:")[-1].strip()
    except: summary = text[:500]

    # RAG + Gen
    matched = match_thesaurus_terms(text, thesaurus_terms)
    ctx = rag_manager.search(f"{summary} {matched}")
    
    mode = CONFIG["mode"]
    instruction = "navrhni JEDEN presný právny nadpis (Title)" if mode == "title" else "vyber JEDEN kľúčový pojem (Keyword)"
    suffix = "Nadpis:" if mode == "title" else "Kľúčový pojem:"

    prompt_gen = (f"Kontext:\n{ctx[:2000]}\n\nZadanie: Na základe súhrnu a pojmov ({matched}) {instruction}.\n\n"
                  f"Súhrn:\n{summary}\n\n{suffix}")

    try:
        gen_res = generator(prompt_gen, max_new_tokens=64, do_sample=True, temperature=0.3)
        output = gen_res[0]["generated_text"].split(suffix)[-1].strip().split("\n")[0].strip('" ')
    except: output = "ERROR"

    return {
        "file": filename, "mode": mode, "summary": summary,
        "priority_terms": matched, "generated_output": output
    }

# ================= 6. MAIN =================
def main():
    print(f"=== STARTING PIPELINE (Mode: {CONFIG['mode']}) ===")
    print(f"=== Hardware: RTX 4070 Super Detected (Optimizing for 12GB VRAM) ===")
    
    stopwords = load_stopwords(CONFIG["stopwords_path"])
    terms = load_thesaurus(CONFIG["thesaurus_path"], CONFIG["thesaurus_col"], stopwords)
    
    summ_model = load_cpu_model(CONFIG["summ_model_id"])
    gen_model = load_gpu_model(CONFIG["main_model_id"])
    rag = RAGManager()

    if not summ_model or not gen_model: 
        print("[FATAL] Models failed to load.")
        return

    # Web Ingest (Optional)
    rag.ingest_web(["https://www.slov-lex.sk/pravne-predpisy/SK/ZZ/2005/300/"])

    results = []
    files = [f for f in os.listdir(CONFIG["input_dir"]) if f.endswith(('.docx', '.rtf', '.txt'))]
    
    for f in files:
        res = process_file(os.path.join(CONFIG["input_dir"], f), rag, summ_model, gen_model, stopwords, terms)
        if res: results.append(res)

    if results:
        df = pd.DataFrame(results)
        out_name = f"output_{CONFIG['mode']}_{datetime.now().strftime('%Y-%m-%d')}.csv"
        df.to_csv(os.path.join(CONFIG["output_dir"], out_name), index=False, encoding='utf-8-sig')
        print(f"\n[DONE] Saved to {out_name}")
        print(df[["file", "generated_output"]].head())

if __name__ == "__main__":
    main()

USER_AGENT environment variable not set, consider setting it to identify your requests.


=== STARTING PIPELINE (Mode: title) ===
=== Hardware: RTX 4070 Super Detected (Optimizing for 12GB VRAM) ===
[THESAURUS] Loaded 3000 terms.
[CPU] Loading slovak-nlp/mistral-sk-7b...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


[GPU] Loading Qwen/Qwen3-4B-Thinking-2507 on RTX 4070 Super...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


[EMB] Loading Embeddings on CPU...
[WEB] Ingested 1 chunks.
--- Processing: 117694.docx ---
